## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [18]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 1
# To reduce runtime, you will sample 20% of the interaction data from user_artist_data.txt using seed=314 (and use this seed throughout the lab)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Before doing anything, create a Spark session.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Define the schema for the file to import.  
# -- I added this after reviewing the data file and running an initial import.
# -- Even though I saw the file had no headers, I didn't initially stop to think I needed to add them.  :)
from pyspark.sql.types import StructType, StructField, IntegerType
autoscrobbler_schema = StructType([
                                    StructField("userid", IntegerType(), False),
                                    StructField("artistid", IntegerType(), False),
                                    StructField("playcount", IntegerType(), False),
                                    ])
     

# Load the data.
# It's tab-delimited so need to include sep=" " since the data are separated by spaces.
df_user = spark.read.csv("/standard/ds7200-apt4c/large_datasets/user_artist_data.txt",  sep=" ", header=False, schema=autoscrobbler_schema)

# Select a 20% sample from the data.
df_user_small = df_user.sample(fraction=0.20, seed=314)

# Confirm rows were imported.
print(f"""There are {df_user_small.count()} rows in the imported data set.""")

# Review a few records and the schema to make sure it was applied correctly.
# -- I had trouble getting this part to work.
df_user.show(5)
df_user.printSchema()

There are 4861443 rows in the imported data set.
+-------+--------+---------+
| userid|artistid|playcount|
+-------+--------+---------+
|1000002|       1|       55|
|1000002| 1000006|       33|
|1000002| 1000007|        8|
|1000002| 1000009|      144|
|1000002| 1000010|      314|
+-------+--------+---------+
only showing top 5 rows
root
 |-- userid: integer (nullable = true)
 |-- artistid: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [19]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 2
# Next, remove users that listened to fewer than 10 artists (you may ignore aliasing in this step)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Remove users that listen to fewer than 10 artists using SQL.
# -- First register the DF as a temp view first.
df_user_small.createOrReplaceTempView("df_user_small")

# See how may users have listened to fewer than 10 artists.

# Select the columns needed for this analysis.
df_user_go = spark.sql("""
SELECT *
FROM df_user_small
WHERE userid IN
    (
        SELECT userid
        FROM df_user_small
        GROUP BY userid
        HAVING COUNT(DISTINCT artistid) > 9
    )
""")

print(f"""There are {df_user_go.count()} rows after removing users that listened to less than 10 artists.""")

[Stage 112:>                                                        (0 + 8) / 8]

There are 4685471 rows after removing users that listened to less than 10 artists.


In [21]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 3
# Lastly, split the data: 60% for training / 20% for validation / 20% for testing
# ----------------------------------------------------------------------------------------------------------------------------------------------
# Split the dataset.
trainDF, valDF, testDF = df_user_go.randomSplit([.6, .2, .2], seed=314)
print(f"""There are {trainDF.count()} rows in the training set, {valDF.count()} rows in the validation set, and {testDF.count()} in the test set.""")

[Stage 153:=================================================>       (7 + 1) / 8]

There are 2809638 rows in the training set, 938020 rows in the validation set, and 937813 in the test set.


In [22]:
# TASK 1 (1 PT)
# Show the first row from the training set
trainDF.show(1)

[Stage 162:=====================>                                   (3 + 5) / 8]

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   321|      40|        1|
+------+--------+---------+
only showing top 1 row


In [23]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
testDF.show(1)

[Stage 170:=====================>                                   (3 + 5) / 8]

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   321|     357|        3|
+------+--------+---------+
only showing top 1 row


In [1]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01


In [3]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1


In [4]:
# TASK 4 (1 PT)
# Briefly explain Recall@K in the context of recommender systems


In [5]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1


In [24]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam, alpha) to maximize Recall@K


In [6]:
# TASK 6 (2 PTS)
# Report the results including best values for (rank, maxIter, regParam, alpha) along with the Recall@K from the validation set


In [27]:
# Retrain the best configuration on train + validation data.


In [7]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.


In [8]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?
